In [22]:
import torchinfo
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
import time
from datasets import load_dataset, Dataset
from nltk.tokenize import sent_tokenize
import random
import pandas as pd
from tqdm import tqdm
import lovely_tensors as lt

lt.monkey_patch()

In [23]:
checkpoint = "HuggingFaceTB/SmolLM2-360M"
device = "mps" # for GPU usage or "cpu" for CPU usage
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

In [3]:
inputs = tokenizer.encode("Gravity is", return_tensors="pt").to(device)
start_time = time.time()
outputs = model.generate(inputs, do_sample=True)
print(f"Execution time: {time.time() - start_time:.2f} seconds")
print(tokenizer.decode(outputs[0]))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Execution time: 8.57 seconds
Gravity is something that needs to be explored thoroughly. In this article by NASA, there’s very little on


# Dataset Creating

In [3]:
ds = load_dataset("HuggingFaceTB/everyday-conversations-llama3.1-2k")

In [4]:
train_split = ds["train_sft"].remove_columns([col for col in ds["train_sft"].column_names if col not in ["completion", "messages"]])
test_split = ds["test_sft"].remove_columns([col for col in ds["test_sft"].column_names if col not in ["completion", "messages"]])

In [19]:
# nltk.download('punkt', 'punkt_tab')
chance_to_remove_end = 0.3  # Probability of removing the symbol at the end of a sentence such as '.!?'

# Flatten messages into a new dataframe
def create_dataframe(split: Dataset):
    messages_flat = set()
    for chat in split["messages"]:
        for message in chat:
            sentences = sent_tokenize(message["content"])
            for sentence in sentences:
                assert(sentence[-1] not in ","), f"Invalid sentence: {sentence}"
                messages_flat.add(sentence)
                random_chance = random.random()
                # delete the symbol at the end of the sentence 30% of the time
                if sentence[-1] in ".!?" and random_chance < chance_to_remove_end:
                    sentence = sentence[:-1]
                messages_flat.add((sentence, 1))
                words = sentence.split()
                if len(words) > 2:  # Ensure at least one word remains
                    num_words_to_remove = random.randint(1, len(words) - 2)
                    truncated_sentence = " ".join(words[:-num_words_to_remove])
                    messages_flat.add((truncated_sentence, 0))
    df = pd.DataFrame(list(messages_flat), columns=["sentence"])
    return df


In [20]:
create_dataframe(train_split).to_csv("data/train_split.csv", index=False)
create_dataframe(test_split).to_csv("data/test_split.csv", index=False)

# Load Datasets and Shuffle if necessary

In [13]:
def load_dataframe_shuffle(df_path: str, shuffle: bool = True) -> Dataset:
    df = pd.read_csv(df_path)
    if shuffle:
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)  # Shuffle the dataset
    return Dataset.from_pandas(df)

In [21]:
train_df = pd.read_csv("data/train_split.csv")
test_df = pd.read_csv("data/test_split.csv")

print("Longest tokenization in train_data:", tokenizer.encode(train_df.loc[train_df['sentence'].str.len().idxmax(), 'sentence'], return_tensors="pt").shape[1], "tokens")
print("Longest tokenization in test_data:", tokenizer.encode(test_df.loc[test_df['sentence'].str.len().idxmax(), 'sentence'], return_tensors="pt").shape[1], "tokens")

Longest tokenization in train_data: 51 tokens
Longest tokenization in test_data: 49 tokens


In [6]:
train_data = load_dataframe_shuffle("train_split_hard.csv" )
test_data = load_dataframe_shuffle("test_split_hard.csv", shuffle=False)

In [7]:
train_data, test_data

(Dataset({
     features: ['sentence', 'eos_label'],
     num_rows: 35140
 }),
 Dataset({
     features: ['sentence', 'eos_label'],
     num_rows: 2141
 }))

In [75]:
random_index = random.randint(0, len(test_data) - 1)
example_sentence, label = test_data[random_index]["sentence"], test_data[random_index]["eos_label"]
print(example_sentence, "\n", f"label: {label}")
inputs = tokenizer.encode(example_sentence, return_tensors="pt").to(device)

What are the basic things I need to get started? 
 label: 1


In [102]:
with torch.no_grad():
    outputs = model(inputs)
    # print(tokenizer.decode(outputs[0]))
    logits = outputs.logits[:, -1, :]
    probabilities = F.softmax(logits, dim=-1)
    
    # Check probability of EOS token
    eos_token_id = tokenizer.eos_token_id
    eos_prob = probabilities[0, eos_token_id].item()
    print(f"EOS Token Probability: {eos_prob:.4f}")

    # Get top 5 most probable tokens
    top_k = 5
    probs, indices = torch.topk(probabilities, top_k)
    for i in range(top_k):
        token = tokenizer.decode(indices[0, i].item())
        prob = probs[0, i].item()
        print(repr(f"Token:{token}, token_id: {indices[0, i].item()} Probability: {prob:.6f}"))

EOS Token Probability: 0.0013
'Token:\n, token_id: 198 Probability: 0.684922'
'Token:\n\n, token_id: 1116 Probability: 0.058103'
'Token:\n\n  , token_id: 39892 Probability: 0.047171'
'Token:\n  , token_id: 11181 Probability: 0.031195'
'Token:\n\n , token_id: 8866 Probability: 0.028721'
'Token:\n , token_id: 3805 Probability: 0.019516'
'Token: What, token_id: 1812 Probability: 0.011018'
'Token:\n\n   , token_id: 1004 Probability: 0.010916'
'Token:\xa0, token_id: 15442 Probability: 0.010250'
'Token: I, token_id: 339 Probability: 0.007934'
'Token:\n\n\n, token_id: 16506 Probability: 0.007047'
'Token: How, token_id: 1073 Probability: 0.005730'
'Token: (, token_id: 365 Probability: 0.004572'
'Token:\n\n\n\n, token_id: 22342 Probability: 0.003817'
'Token:\n   , token_id: 472 Probability: 0.002709'
'Token: , token_id: 216 Probability: 0.002231'
'Token: �, token_id: 3351 Probability: 0.002220'
'Token:\n\n    , token_id: 35988 Probability: 0.002036'
'Token: Do, token_id: 3315 Probability: 0.002

# Special end of sentence tokens

In [8]:
eos_tokens = ["<|endoftext|>", ".", "?", "!", ";", "\n", "\n\n", "\n\n\n", "\n\n\n\n", ".\"", "\xa0", ".”", ".)",
              ".,", ".\\\\", ".;", "\n ", "\n  ", "\n  ", "\n\n  ", "\n\n   ","\n\n    "]

In [9]:
eos_ids = []
for tok in eos_tokens:
    eos_ids.append(tokenizer.encode(text=tok)[0])
print(eos_ids)

[0, 30, 47, 17, 43, 198, 1116, 16506, 22342, 1270, 15442, 1184, 2677, 1143, 30, 5282, 3805, 11181, 11181, 39892, 1004, 35988]


# Baseline Evaluation

In [10]:
batch_size = 64

# Prepare test data for model evaluation
test_sentences = test_data["sentence"]
test_labels = torch.tensor(test_data["eos_label"], dtype=torch.long).to(device)
data_loader = DataLoader(test_sentences, batch_size=batch_size, shuffle=False)

In [11]:
tokenizer.pad_token = tokenizer.eos_token

In [12]:
probabilities_list = []

# Run model inference in batches
for batch in tqdm(data_loader, desc="Processing batches"):
    inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True).to(device)
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # Shape: [batch_size, seq_length, vocab_size]

    # Find the actual last token index per sentence
    last_token_indices = attention_mask.sum(dim=1) - 1  # Last actual token index before padding

    # Gather the correct logits for each sentence’s last token
    batch_size_actual = logits.shape[0]
    last_token_logits = logits[torch.arange(batch_size_actual), last_token_indices, :]  # Shape: [batch_size, vocab_size]

    # Compute softmax to get probabilities
    probabilities = F.softmax(last_token_logits, dim=-1)

    # Sum probabilities for all tokens in eos_ids
    eos_probs = probabilities[:, eos_ids].sum(dim=-1)
    probabilities_list.extend(eos_probs.cpu().numpy())

Processing batches:   0%|          | 0/34 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Processing batches: 100%|██████████| 34/34 [00:32<00:00,  1.04it/s]


In [13]:
test_df = test_data.to_pandas()

test_df["probability"] = probabilities_list

In [14]:
thresholds = [0.15, 0.3, 0.5, 0.7]
accuracies = {}

total_eos_samples = (test_df["eos_label"] == 1).sum()
assert total_eos_samples > 0

for threshold in thresholds:
    correct_predictions = (((test_df["eos_label"] == 1) & (test_df["probability"] >= threshold))
                            | ((test_df["eos_label"] == 0) & (test_df["probability"] < threshold))).sum()
    print(f"Accuracy@{int(threshold * 100)}: {((correct_predictions / test_df.shape[0]) * 100.0):.2f}%")

Accuracy@15: 89.82%
Accuracy@30: 80.06%
Accuracy@50: 69.50%
Accuracy@70: 60.95%


In [19]:
test_df = test_df.rename(columns={'probability': 'baseline_prob'})
test_df.to_csv("test_results_baseline.csv", index=False)